# 10 · Emotion Mirror: Optional Multimodal Comparison

This notebook documents the experimental extension of the voice-first website. Its research question is whether **reviewed text, visible expression and vocal delivery yield different model outputs for the same utterance**. The interface name “Mind-reading” is metaphorical; no inference about hidden feelings is justified.

The five local text models and selected DeepSeek prompt are retained from the earlier experiments. The two new models are independently selected, frozen and evaluated; they are not fused with the text probabilities.

## 1. Reproducible artifacts

Model downloads and inference are explicit preparation steps. This notebook inspects the recorded evaluation and verifies model hashes without retraining, downloading data, recording participants, or making paid API requests.

In [1]:
from pathlib import Path
import json, hashlib
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'api').exists(): ROOT = ROOT.parent
assert (ROOT / 'api/multimodal.py').exists(), 'Open from the project or notebooks directory.'
manifest = json.loads((ROOT / 'docs/reproducibility/multimodal_models.json').read_text())
report = json.loads((ROOT / 'docs/reproducibility/multimodal_validation.json').read_text())
rows = []
for channel, model in manifest.items():
    for artifact in model['files']:
        path = ROOT / 'models/multimodal' / channel / artifact['file']
        with path.open('rb') as handle:
            assert hashlib.file_digest(handle, 'sha256').hexdigest() == artifact['sha256']
    rows.append({'channel':channel, 'repository':model['repository'], 'revision':model['revision'], 'license':model['license']})
display(pd.DataFrame(rows))

,channel,repository,revision,license
0,face,trpakov/vit-face-expression,ef0bc6fc34241b6587e7e009e7711357be28c024,Apache-2.0
1,acoustic,superb/wav2vec2-base-superb-er,441a7599c3b22107314dcbd9166621c5c83f2cc5,Apache-2.0


## 2. Independent selection and inference contracts

The expression model is a FER2013-trained ViT with seven native labels. The acoustic model is a SUPERB/IEMOCAP wav2vec 2.0 checkpoint with four native labels. They were selected for documented preprocessing and licenses, frozen weights, and CPU availability; no comprehensive superiority claim is made.

Frontal-face detection precedes classification. Up to three valid frame distributions are averaged. Acoustic inference uses 16 kHz mono audio and duration-weighted distributions over contiguous chunks of at most eight seconds. Pitch and energy are descriptive signal measurements, not rules for emotion assignment. Neither new model is calibrated to the course dataset.

In [2]:
from api.multimodal import FACE_LABELS, AUDIO_LABELS
from src.config import LABELS
display(pd.DataFrame([
    {'channel':'text', 'labels':', '.join(LABELS), 'probability calibration':'Saved validation temperature, where applicable; DeepSeek N/A'},
    {'channel':'expression', 'labels':', '.join(FACE_LABELS), 'probability calibration':'None'},
    {'channel':'acoustic', 'labels':', '.join(AUDIO_LABELS), 'probability calibration':'None'},
]))

,channel,labels,probability calibration
0,text,"sadness, joy, love, anger, fear, surprise","Saved validation temperature, where applicable..."
1,expression,"angry, disgust, fear, happy, neutral, sad, sur...",None
2,acoustic,"neu, hap, ang, sad",None


## 3. External smoke evaluation

The predeclared convenience sample consists of CREMA-D actors 1001–1006 speaking sentence DFA at intensity XX. Expression covers six acted classes (36 clips); acoustic covers four supported classes (24 clips). Intended acting labels come from filenames, rather than crowd-perceived majority judgments. These are neither inner feelings nor deployment ground truth. No threshold, model weight or example was tuned using these results.

The raw fixtures are ignored by Git. The report records the dataset commit, file digests, every prediction, and failures. The source is [Cao et al. (2014), CREMA-D](https://github.com/CheyneyComputerScience/CREMA-D), ODbL-1.0 / DbCL-1.0.

In [3]:
summary = []
for channel in ['face','acoustic']:
    r=report[channel]
    summary.append({'channel':channel, 'attempted':r['attempted'], 'predicted':r['predicted'],
                    'coverage':r['coverage'], 'accuracy_including_failures':r['accuracy_counting_failures_incorrect'],
                    'accuracy_on_predicted':r['accuracy_on_predicted'],
                    'macro_f1_on_predicted':r['macro_f1_on_predicted_represented_classes']})
display(pd.DataFrame(summary).round(4))
for channel in ['face','acoustic']:
    r=report[channel]
    print(channel, '— rows: intended; columns: predicted')
    display(pd.DataFrame(r['confusion_matrix'],index=r['confusion_labels'],columns=r['confusion_labels']))

,channel,attempted,predicted,coverage,accuracy_including_failures,accuracy_on_predicted,macro_f1_on_predicted
0,face,36,35,0.9722,0.4444,0.4571,0.3937
1,acoustic,24,23,0.9583,0.4167,0.4348,0.3422


face — rows: intended; columns: predicted


,angry,disgust,fear,happy,neutral,sad,surprise
angry,2,0,0,2,1,0,0
disgust,3,1,1,0,1,0,0
fear,1,0,0,0,3,0,2
happy,0,0,0,6,0,0,0
neutral,0,0,0,0,6,0,0
sad,1,0,0,0,3,1,1
surprise,0,0,0,0,0,0,0


acoustic — rows: intended; columns: predicted


,ang,hap,neu,sad
ang,6,0,0,0
hap,6,0,0,0
neu,6,0,0,0
sad,1,0,0,4


## 4. Error inspection and interpretation

Expression is correct on 16/36 attempts and acoustic on 10/24. Each channel abstains once. Expression struggles with fear and sadness; the acoustic checkpoint frequently assigns anger to happy and neutral examples. This is evidence of limited transfer, not successful mind reading.

The expression matrix includes a surprise column although CREMA-D provides no surprise reference examples. Macro-F1 is restricted to represented reference classes. Accuracy counting failures as incorrect and coverage prevent abstentions from being hidden. Six actors and one sentence do not establish fairness, spontaneous-speech performance, or phone-microphone reliability.

In [4]:
predictions = pd.DataFrame(report['predictions'])
errors = predictions[predictions['prediction'].isna() | (predictions['gold'] != predictions['prediction'])]
display(errors[['clip','channel','gold','prediction','error']].fillna('—'))

,clip,channel,gold,prediction,error
2,1001_DFA_DIS_XX,face,disgust,angry,—
3,1001_DFA_FEA_XX,face,fear,angry,—
5,1001_DFA_HAP_XX,acoustic,hap,ang,—
7,1001_DFA_NEU_XX,acoustic,neu,ang,—
8,1001_DFA_SAD_XX,face,sad,angry,—
9,1001_DFA_SAD_XX,acoustic,sad,—,Insufficient voiced signal. Try a clear senten...
10,1002_DFA_ANG_XX,face,angry,happy,—
12,1002_DFA_DIS_XX,face,disgust,angry,—
13,1002_DFA_FEA_XX,face,fear,surprise,—
15,1002_DFA_HAP_XX,acoustic,hap,ang,—


## 5. Application integration and validation boundary

The bonus is initially off. Users independently enable expression and/or acoustic analysis, record a new utterance, review its transcript and compare. The recording replaces earlier text, and snapshots share its capture interval. Text, expression and audio outputs remain separate. Editing the transcript does not alter captured media.

Recordings are sent for Whisper transcription after stopping. Only the selected bonus channels are sent when Compare is pressed; the API rejects unselected media. No captured audio or image is written to disk by the application. Public hosting changes the processing location from the user's computer to the project server. Optional DeepSeek receives only confirmed text.

Automated checks cover malformed requests, size bounds, no/multiple face handling, silence, signal features, busy service, partial success and usage controls. The integration example below uses public fixtures through the same HTTP contracts. It does not replace manual live microphone/camera acceptance on presentation devices.

In [5]:
integration = json.loads((ROOT / 'docs/reproducibility/space_local_smoke.json').read_text())
print('Fixture:', integration['fixture'])
print('Whisper transcript:', integration['transcript']['text'])
display(pd.DataFrame(
    [{'channel':'text', 'model':r['model_id'], 'label':r['label']} for r in integration['text']['results']] +
    [{'channel':r['channel'], 'model':r['model'], 'label':r['label']} for r in integration['multimodal']['results']]
))
assert not integration['text']['errors'] and not integration['multimodal']['errors']

Fixture: 1001_DFA_HAP_XX
Whisper transcript: Don't forget to check it


,channel,model,label
0,text,tfidf_lr,joy
1,text,bert,joy
2,face,trpakov/vit-face-expression,happy
3,acoustic,superb/wav2vec2-base-superb-er,ang


## 6. Deployment and remaining acceptance

The public package retains the same React design, exports static assets, serves same-origin prediction routes through FastAPI, and mounts a Gradio service inspector. Only allowlisted source, model artifacts and evaluation summaries are uploaded. Deployment credentials, `.env`, participant media, raw datasets, notebooks and backups are excluded.

Shared request and body limits bound demo resource use. DeepSeek has a per-process call allowance, not a monetary billing cap. Hugging Face availability and hardware eligibility are external dependencies. A public URL should be reported as complete only after its running service and actual prediction routes have been tested. Final live microphone/camera testing on the group's own desktop and phone remains a separate acceptance item.